### Work with the Survey Manager

In [ ]:
import arcgis
from arcgis.gis import GIS
import datetime
from datetime import date, timedelta
import shutil
import os
gis = GIS(username="survey123_autotest_creator", password="Autotest123")

In [ ]:
survey_manager = arcgis.apps.survey123.SurveyManager(gis)
surveys = survey_manager.surveys
assert len(surveys) > 0

In [ ]:
survey_by_id = survey_manager.get("2ec66822b5364e1fa6e64f4ac7581d2e")
assert len(survey_by_id.properties['title']) > 0

In [ ]:
forms = gis.content.search('type:form owner:survey123_autotest_creator')
assert len(forms) > 0

In [ ]:
survey_by_item = survey_manager.get(forms[8])
assert len(survey_by_item.properties) > 0

### Work with survey data

In [ ]:
# Download - file formats
dl_formats = ['CSV', 'Shapefile', 'File Geodatabase']
for f in dl_formats:
    outfile = survey_by_id.download(f)
    assert outfile != None

In [ ]:
# Download - pandas DataFrame
import pandas as pd
survey_df = survey_by_id.download('DF')
assert len(survey_df) > 0

### Create reports


In [ ]:
# Identify report templates associated with a survey
templates = survey_by_id.report_templates
assert len(templates) > 0

In [ ]:
# Generate a default report template 
temp_name = "Sample template " + str(datetime.datetime.now().strftime("%Y%m%d%H%M%S"))
new_template = survey_by_id.create_report_template(template_name=temp_name)
assert new_template != None

In [ ]:
# Check template syntax
check = survey_by_id.check_template_syntax(new_template)
assert check['success'] == True

In [ ]:
# Associate a report template with a survey
upload_template = survey_by_id.upload_report_template(template_file=new_template, template_name="PythonAPITemplate")
assert upload_template != None
templates = survey_by_id.report_templates
updated_templates = [x.title for x in templates]
assert 'PythonAPITemplate' in updated_templates

In [ ]:
# Estimate credits
credits = survey_by_id.estimate(templates[0], where="1=1")
assert credits['success'] == True

In [ ]:
# Create sample report
webmap = gis.content.search(query="title:Water Quality Inspection Python API", item_type="Web Map")
wm_item=webmap[0]
sample = survey_by_id.create_sample_report(templates[0], where="objectid=1", utc_offset="-07:00", 
                                           report_title="Sample_Report", merge_files="none", survey_item=survey_by_id, 
                                           webmap_item=wm_item, map_scale="10000", locale='en')
assert sample != None

In [ ]:
# Generate single report
report = survey_by_id.generate_report(templates[0], where="objectid=122")
assert report != None

In [ ]:
# Generate multiple reports
local_batch_reports = survey_by_id.generate_report(templates[0], where="ws_advisory = 'Yes' and ws_advisory_start_date > '01/01/2020'", 
                                                   report_title="SingleReportInstance", package_name="ReportPackageNamePython")
assert local_batch_reports != None

In [ ]:
# Generate multiple reports and save to your orgainzation
nowstring = datetime.datetime.now().strftime("%Y%m%d%H%M%S")
folder_ID = survey_by_id.properties['ownerFolder']
org_batch_reports = survey_by_id.generate_report(templates[0], where="ws_advisory = 'Yes' and ws_advisory_start_date > '01/01/2020'",
                                                 package_name="Test_{0}".format(nowstring), folder_id=folder_ID)
assert org_batch_reports != None
search_for_batch_reports = gis.content.search("Test_{0}".format(nowstring))
assert len(search_for_batch_reports) > 0

In [ ]:
# Generate report with all possible parameters
folder_ID = survey_by_id.properties['ownerFolder']
webmap = gis.content.search(query="title:Water Quality Inspection Python API", item_type="Web Map")
wm_item=webmap[0]
all_params = survey_by_id.generate_report(templates[0], where="ws_advisory = 'Yes' and ws_advisory_start_date > '01/01/2020'", utc_offset="-07:00", report_title="All_Params_Report",
                                          package_name="All_Params_Package", output_format="pdf", folder_id=folder_ID, 
                                          merge_files="none", survey_item=survey_by_id, webmap_item=wm_item, 
                                          map_scale="10000", locale="en")
assert all_params != None

In [ ]:
# Update report template
import tempfile
template_location = os.path.join(os.path.abspath(""), "Survey123_resources", "Sample_template.docx")
tmpdir = tempfile.TemporaryDirectory()
tmp_folder = tmpdir.name
updated_template = shutil.copy(template_location, os.path.join(tmp_folder, 'PythonAPITemplate.docx'))
update = survey_by_id.update_report_template(updated_template)
assert len(update) > 0

In [ ]:
recentReports = survey_by_id.reports
assert len(recentReports) > 0

### Create and publish Surveys

In [ ]:
new_survey = survey_manager.create(
    title="Python Test Survey",
    tags="ArcGIS API for Python, Survey123, Form",
    summary="This survey was created using the ArcGIS API for Python",
)
assert type(new_survey) == arcgis.apps.survey123.Survey

In [ ]:
published_survey = new_survey.publish(
    xlsform=os.path.join(os.path.abspath(""), "Survey123_resources", "Hydrant_Inspection_init.xlsx"),
    info={
        "queryInfo": {
            "mode": "manual",
            "editEnabled": True,
            "copyEnabled": True
        },
        "sentInfo": {
            "enabled": True,
            "editEnabled": True,
            "copyEnabled": True
        },
        "displayInfo": {
            "map": {
                "coordinateFormat" : "usng",
                "home": {
                    "latitude": 34.0568,
                    "longitude": -117.1961,
                    "zoomLevel": 20
                },
                "preview": {
                    "coordinateFormat": "usng",
                    "zoomLevel": 0
                }
            }
        }
    },
    create_web_form = True,
    enable_delete_protection = False,
    create_coded_value_domains = True,
    enable_sync = False,
    create_web_map = True
)

assert type(published_survey) == arcgis.apps.survey123.Survey

usr = arcgis.gis.User(gis, gis.users.me.username)
full_folder = usr.folders
survey_folder = next((f for f in full_folder if f['id'] == published_survey.properties['ownerFolder']), 0)
fldr_items = usr.items(folder=survey_folder)
fldr_items_types = [x.type for x in fldr_items]

assert "Form" in fldr_items_types
assert "Feature Service" in fldr_items_types
assert "Web Map" in fldr_items_types

for item in fldr_items:
    if item.type == "Feature Service" and item == [x for x in fldr_items if x.type == "Form"][0].related_items('Survey2Service','forward')[0]:
        assert "View Service" in item.typeKeywords
        

In [ ]:
updated_survey = published_survey.publish(
    xlsform=os.path.join(os.path.abspath(""), "Survey123_resources", "Hydrant_Inspection_update.xlsx"),
    schema_changes=True
)

assert type(updated_survey) == arcgis.apps.survey123.Survey
sub_url = [x for x in fldr_items if x.type == "Form"][0].related_items('Survey2Service','forward')[0]
assert 'defects' in [x.properties.name for x in sub_url.tables]


### Create Survey webhooks

In [ ]:
# Ensure there are no webhooks currently configured

assert len(updated_survey.webhooks) == 0

In [ ]:
# Add a webhook

add_result = updated_survey.add_webhook(
    name="Python Test Webhook", 
    payload_url="https://www.arcgis.com", 
    trigger_events=['addData', 'editData'],
    portal_info=True,
    submitted_record=True,
    user_info=True,
    server_response=True,
    survey_info=True,
    active=True)

assert add_result['success'] is True
assert len(updated_survey.webhooks) > 0

added_webhook = [x for x in updated_survey.webhooks if x['id'] == add_result['webhookId']][0]

assert added_webhook['active'] is True
assert added_webhook['name'] == "Python Test Webhook"
assert added_webhook['url'] == "https://www.arcgis.com"
assert added_webhook['includePortalInfo'] is True
assert added_webhook['includeServiceRequest'] is True
assert added_webhook['includeUserInfo'] is True
assert added_webhook['includeServiceResponse'] is True
assert added_webhook['includeSurveyInfo'] is True
assert added_webhook['events'] == ['addData', 'editData']

In [ ]:
# Update a webhook

update_result = updated_survey.update_webhook(
    webhook_id = add_result['webhookId'],
    name = "Python Test Webhook Updated",
    portal_info = False,
    user_info = False,
    survey_info = False
)

updated_webhook = [x for x in updated_survey.webhooks if x['id'] == update_result['webhookId']][0]

assert updated_webhook['active'] is True
assert updated_webhook['name'] == "Python Test Webhook Updated"
assert updated_webhook['url'] == "https://www.arcgis.com"
assert updated_webhook['includePortalInfo'] is False
assert updated_webhook['includeServiceRequest'] is True
assert updated_webhook['includeUserInfo'] is False
assert updated_webhook['includeServiceResponse'] is True
assert updated_webhook['includeSurveyInfo'] is False
assert updated_webhook['events'] == ['addData', 'editData']


In [ ]:
# Delete a webhook

delete_result = updated_survey.delete_webhook(update_result['webhookId'])
assert delete_result is True

assert len(updated_survey.webhooks) == 0

In [ ]:
assert upload_template.delete() == True
assert org_batch_reports.delete() == True
assert all_params.delete() == True
assert [x for x in fldr_items if x.type == "Web Map"][0].delete() == True
assert [x for x in fldr_items if x.type == "Form"][0].delete() == True
assert [x for x in fldr_items if x.title == "Python Test Survey_form"][0].delete() == True
assert gis.content.delete_folder("Survey-Python Test Survey") == True
